# The Segmentation Detective
## An Interactive XAI Journey Through Pedestrian Infrastructure Mapping

---

> *"Every pixel tells a story. But can we trust what the model claims to see?"*

This interactive notebook lets you **investigate** how Tile2Net's deep learning model detects sidewalks, roads, and crosswalks from aerial imagery. Upload your own image or use our sample tiles, then watch as 4 distinct **visual languages** reveal the model's inner workings.

### What You'll Discover
- **Cartoon Blocks**: The model's confident claims
- **Thermal Fog**: Where uncertainty lurks
- **Glowing Ghost**: What catches the model's attention
- **Static Noise**: Raw pixel-level evidence

---

**Team:** Swapnil Sharma, Uttam Singh, Adamay Mann  
**Course:** CS-GY 9223 Visualization for Machine Learning (Fall 2024)  
**NYU Tandon School of Engineering**

---
## Section 1: Environment Setup

First, let's set up our investigation toolkit.

In [ ]:
# 1.1 Clone repository and install dependencies
!git clone https://github.com/mannadamay12/tile2net-segD
%cd tile2net-segD
!pip install -e . -q
!pip install grad-cam captum saliency ipywidgets -q

print("Dependencies installed!")

In [ ]:
# 1.2 Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from torchvision import transforms
import cv2
from scipy.ndimage import sobel
import os
import gc
from collections import OrderedDict
from io import BytesIO
import requests
import zipfile
import math

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n All imports successful!")

In [ ]:
# 1.3 Load the Tile2Net model

# FIX: Add the nested directory to Python path
import sys
import os

# The repo clones into tile2net-segD/tile2net-segD/ (nested)
# Find the correct src path
possible_paths = [
    '/kaggle/working/tile2net-segD/tile2net-segD/src',
    '/kaggle/working/tile2net-segD/src',
    './tile2net-segD/src',
    './src'
]

for path in possible_paths:
    if os.path.exists(path):
        sys.path.insert(0, path)
        print(f"Added to path: {path}")
        break

# Now import tile2net
from tile2net.tileseg.network.ocrnet import MscaleOCR
from tile2net.tileseg.config import cfg
from tile2net.namespace import torch_version_float

# Paths to model weights (update if needed)
HRNET_CHECKPOINT = '/kaggle/input/tile2nest/pytorch/default/1/hrnetv2_w48_imagenet_pretrained.pth'
SEG_CHECKPOINT = '/kaggle/input/tile2nest/pytorch/default/1/satellite_2021.pth'

# Configure model
cfg.MODEL.ARCH = 'ocrnet.HRNet_Mscale'
cfg.DATASET.NUM_CLASSES = 4
cfg.MODEL.OCR.MID_CHANNELS = 512
cfg.MODEL.OCR.KEY_CHANNELS = 256
cfg.MODEL.BNFUNC = torch.nn.BatchNorm2d
cfg.MODEL.HRNET_CHECKPOINT = HRNET_CHECKPOINT
cfg.MODEL.MSCALE = True
cfg.MODEL.MSCALE_LO_SCALE = 0.5
cfg.OPTIONS.TORCH_VERSION = torch_version_float()

# Load model
print("Loading model...")
net = MscaleOCR(num_classes=4, trunk='hrnetv2', criterion=None)
checkpoint = torch.load(SEG_CHECKPOINT, map_location='cpu', weights_only=False)
state_dict = checkpoint['state_dict']

new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k[7:] if k.startswith('module.') else k
    new_state_dict[name] = v

net.load_state_dict(new_state_dict)
net = net.cuda().eval()

print("Model loaded successfully!")
print(f"  Architecture: MscaleOCR (HRNet-W48)")
print(f"  Classes: 4 (Sidewalk, Road, Crosswalk, Background)")

In [ ]:
# 1.4 Create model wrapper and define constants

class ModelWrapper(nn.Module):
    """Wrapper to make MscaleOCR compatible with pytorch-grad-cam"""
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.backbone = model.backbone

    def forward(self, x):
        output = self.model({'images': x})
        if isinstance(output, dict):
            return output.get('pred', output.get('out', list(output.values())[0]))
        return output

wrapped_net = ModelWrapper(net)
wrapped_net.eval()

# Constants
CLASS_NAMES = ['Sidewalk', 'Road', 'Crosswalk', 'Background']
CLASS_COLORS = ['cornflowerblue', 'dimgray', 'gold', 'lightgray']
CARTOON_COLORS = {
    0: (74, 144, 217),   # Sidewalk - Blue
    1: (45, 45, 45),     # Road - Dark Gray
    2: (245, 166, 35),   # Crosswalk - Amber
    3: (232, 232, 232),  # Background - Light Gray
}

# Preprocessing
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Model wrapper created!")
print(f"  Classes: {CLASS_NAMES}")

---
## Section 2: Choose Your Investigation Target

Upload your own aerial image or select from our sample tiles.

In [ ]:
# 2.1 Sample Tiles - Download helper functions

def lat_lon_to_tile(lat, lon, zoom):
    """Convert lat/lon to tile coordinates"""
    n = 2.0 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n)
    return x, y

def download_tile_arcgis(lat, lon, zoom=19, size=512):
    """Download tile from ArcGIS (for NYC)"""
    tile_x, tile_y = lat_lon_to_tile(lat, lon, zoom)
    tiles_per_side = size // 256
    
    img = Image.new('RGB', (size, size))
    for dy in range(tiles_per_side):
        for dx in range(tiles_per_side):
            url = f"https://tiles.arcgis.com/tiles/yG5s3afENB5iO9fj/arcgis/rest/services/NYC_Orthos_2024/MapServer/tile/{zoom}/{tile_y+dy}/{tile_x+dx}"
            try:
                resp = requests.get(url, timeout=30)
                if resp.status_code == 200:
                    tile = Image.open(BytesIO(resp.content)).convert('RGB')
                    img.paste(tile, (dx*256, dy*256))
            except:
                pass
    return img

# Sample tile configurations
SAMPLE_TILES = {
    'Boston (Example)': {
        'type': 'generated',
        'path': '/tmp/tile2net/example/tiles/stitched/256_19_4',
        'description': 'Boston area - residential with water'
    },
    'Washington Square Park (NYC)': {
        'type': 'arcgis',
        'lat': 40.7308,
        'lon': -73.9973,
        'description': 'NYC - park with paths and roads'
    },
    'City Tile': {
        'type': 'upload',
        'description': 'Urban area with roads and buildings'
    },
    'Desert Tile': {
        'type': 'upload',
        'description': 'Arid region with sparse infrastructure'
    },
    'Satellite Tile': {
        'type': 'upload',
        'description': 'High-resolution satellite imagery'
    }
}

print(f"Available sample tiles: {list(SAMPLE_TILES.keys())}")

In [ ]:
# 2.2 Interactive Image Selection Widget

# Global state
current_image = None
current_name = "investigation"
output_dir = None

# Create output widget for displaying results
output_area = widgets.Output()

# Dropdown for sample tiles
tile_dropdown = widgets.Dropdown(
    options=['-- Select Sample Tile --'] + list(SAMPLE_TILES.keys()),
    value='-- Select Sample Tile --',
    description='Sample:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)

# File upload widget
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image',
    layout=widgets.Layout(width='300px')
)

# Output directory name
dir_name_input = widgets.Text(
    value='my_investigation',
    description='Save as:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)

# Load button
load_button = widgets.Button(
    description='Load Image',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='150px')
)

# Status label
status_label = widgets.HTML(value='<i>Select a sample tile or upload your own image</i>')

def load_image(b):
    global current_image, current_name, output_dir
    
    with output_area:
        clear_output(wait=True)
        
        # Check if upload has content
        if upload_widget.value:
            # Load from upload
            uploaded_file = list(upload_widget.value.values())[0]
            current_image = Image.open(BytesIO(uploaded_file['content'])).convert('RGB')
            current_name = uploaded_file['metadata']['name'].split('.')[0]
            status_label.value = f'<b style="color:green;">Loaded uploaded image: {current_name}</b>'
            
        elif tile_dropdown.value != '-- Select Sample Tile --':
            # Load from sample
            tile_config = SAMPLE_TILES[tile_dropdown.value]
            current_name = tile_dropdown.value.replace(' ', '_').lower()
            
            if tile_config['type'] == 'generated':
                # Load from generated tiles (Boston)
                tile_path = Path(tile_config['path'])
                if tile_path.exists():
                    img_files = list(tile_path.glob('*.jpg')) + list(tile_path.glob('*.png'))
                    if img_files:
                        current_image = Image.open(img_files[0]).convert('RGB')
                    else:
                        print("No images found. Running tile generation...")
                        !bash ./examples/example.sh < <(echo "")
                        img_files = list(tile_path.glob('*.jpg'))
                        if img_files:
                            current_image = Image.open(img_files[0]).convert('RGB')
                else:
                    print("Running tile generation for Boston example...")
                    !bash ./examples/example.sh < <(echo "")
                    img_files = list(tile_path.glob('*.jpg'))
                    if img_files:
                        current_image = Image.open(img_files[0]).convert('RGB')
                        
            elif tile_config['type'] == 'arcgis':
                # Download from ArcGIS (NYC)
                print(f"Downloading {tile_dropdown.value}...")
                current_image = download_tile_arcgis(tile_config['lat'], tile_config['lon'])
                
            elif tile_config['type'] == 'upload':
                status_label.value = '<b style="color:orange;">Please upload the tile image using the upload button above</b>'
                return
            
            status_label.value = f'<b style="color:green;">Loaded: {tile_dropdown.value}</b>'
        else:
            status_label.value = '<b style="color:red;">Please select a tile or upload an image</b>'
            return
        
        # Setup output directory
        output_dir = Path(f'./outputs/{dir_name_input.value}')
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output directory: {output_dir}")
        
        # Display preview
        if current_image:
            fig, ax = plt.subplots(1, 1, figsize=(8, 8))
            ax.imshow(current_image)
            ax.set_title(f'Loaded: {current_name} ({current_image.size[0]}x{current_image.size[1]})', fontsize=14, fontweight='bold')
            ax.axis('off')
            plt.tight_layout()
            plt.show()
            print(f"\n Image loaded! Proceed to Section 3 to run inference.")

load_button.on_click(load_image)

# Display widgets
print("="*60)
print("IMAGE SELECTION")
print("="*60)
display(widgets.VBox([
    widgets.HTML('<h4>Option 1: Select Sample Tile</h4>'),
    tile_dropdown,
    widgets.HTML('<h4>Option 2: Upload Your Own</h4>'),
    upload_widget,
    widgets.HTML('<hr>'),
    dir_name_input,
    load_button,
    status_label,
]))
display(output_area)

---
## Section 3: Run Inference

Let's see what the model predicts for your image.

In [ ]:
# 3.1 Run segmentation inference

# Global variables for results
img = None
img_array = None
img_normalized = None
input_tensor = None
pred_logits = None
pred_mask = None
probs = None

def run_inference():
    global img, img_array, img_normalized, input_tensor, pred_logits, pred_mask, probs
    
    if current_image is None:
        print("Please load an image first (Section 2)")
        return False
    
    print("Running inference...")
    
    # Prepare image
    img = current_image
    img_array = np.array(img)
    img_normalized = img_array.astype(np.float32) / 255.0
    input_tensor = preprocess(img).unsqueeze(0).cuda()
    
    # Run model
    with torch.no_grad():
        output = net({'images': input_tensor})
    
    pred_logits = output.get('pred', output.get('out', list(output.values())[0]))
    pred_mask = pred_logits.argmax(dim=1).cpu().numpy()[0]
    probs = F.softmax(pred_logits, dim=1).cpu().numpy()[0]
    
    print(f"  Input shape: {img.size}")
    print(f"  Output shape: {pred_mask.shape}")
    print(f"  Classes detected: {np.unique(pred_mask)}")
    
    return True

def visualize_inference():
    if pred_mask is None:
        print("Run inference first!")
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Original
    axes[0].imshow(img)
    axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # Prediction
    cmap = plt.cm.colors.ListedColormap(['cornflowerblue', 'dimgray', 'gold', 'lightgray'])
    axes[1].imshow(pred_mask, cmap=cmap, vmin=0, vmax=3)
    axes[1].set_title('Segmentation Prediction', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # Class distribution
    axes[2].axis('off')
    axes[2].set_xlim(0, 1)
    axes[2].set_ylim(0, 1)
    for i in range(4):
        pct = (pred_mask == i).sum() / pred_mask.size * 100
        axes[2].text(0.1, 0.85 - i*0.2, f"{CLASS_NAMES[i]}: {pct:.1f}%",
                    fontsize=14, bbox=dict(boxstyle='round', facecolor=CLASS_COLORS[i], 
                    alpha=0.7, edgecolor='black'))
    axes[2].set_title('Class Distribution', fontsize=14, fontweight='bold')
    
    plt.suptitle(f'Base Inference: {current_name}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    # Save
    if output_dir:
        save_path = output_dir / 'base_inference.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()

# Run
if run_inference():
    visualize_inference()
    print("\n Inference complete! Proceed to Section 4 for visual styles.")

---
## Section 4: The 4 Visual Styles

Each style reveals a different aspect of the model's decision-making process.

### Style 1: CARTOON BLOCKS
> *"This is my claim" - The model's confident prediction*

Sharp, solid colors with comic-book edges. No gradients, no uncertainty - just the model's definitive classification of every pixel.

In [ ]:
# 4.1 Style 1: Cartoon Blocks

def create_cartoon_prediction(pred_mask, colors_dict, add_edges=True):
    """Create sharp, solid-color segmentation with comic-book edges."""
    h, w = pred_mask.shape
    cartoon = np.zeros((h, w, 3), dtype=np.uint8)
    
    for class_id, rgb in colors_dict.items():
        mask = pred_mask == class_id
        cartoon[mask] = rgb
    
    if add_edges:
        edges_x = sobel(pred_mask, axis=0)
        edges_y = sobel(pred_mask, axis=1)
        edges = (np.abs(edges_x) + np.abs(edges_y)) > 0
        cartoon[edges] = (0, 0, 0)  # Black outlines
    
    return cartoon

if pred_mask is not None:
    cartoon_pred = create_cartoon_prediction(pred_mask, CARTOON_COLORS, add_edges=True)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(img)
    axes[0].set_title('Original Input', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(cartoon_pred)
    axes[1].set_title('Style 1: CARTOON BLOCKS\n"This is my claim"', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    # Legend
    axes[2].axis('off')
    axes[2].set_xlim(0, 1)
    axes[2].set_ylim(0, 1)
    for i, (class_id, rgb) in enumerate(CARTOON_COLORS.items()):
        color_norm = tuple(c/255 for c in rgb)
        rect = plt.Rectangle((0.1, 0.8 - i*0.18), 0.12, 0.1, 
                             facecolor=color_norm, edgecolor='black', linewidth=2)
        axes[2].add_patch(rect)
        pct = (pred_mask == class_id).sum() / pred_mask.size * 100
        axes[2].text(0.26, 0.85 - i*0.18, f'{CLASS_NAMES[class_id]}: {pct:.1f}%', fontsize=13, va='center')
    axes[2].set_title('Class Legend', fontsize=14, fontweight='bold')
    
    plt.suptitle('Style 1: Cartoon Blocks - Diagrammatic Prediction', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if output_dir:
        save_path = output_dir / 'style1_cartoon_blocks.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
else:
    print("Run inference first (Section 3)")

### Style 2: THERMAL FOG
> *"Where danger lurks" - Uncertainty visualization*

Dark regions indicate confidence; bright orange/white reveals where the model is uncertain. Using Shannon entropy to capture richer variation than simple max-probability.

In [ ]:
# 4.2 Style 2: Thermal Fog (Entropy-based)

def create_thermal_fog(probs, method='entropy'):
    """Create uncertainty visualization using entropy."""
    if method == 'entropy':
        probs_clamped = np.clip(probs, 1e-10, 1.0)
        entropy = -np.sum(probs_clamped * np.log(probs_clamped), axis=0)
        max_entropy = np.log(probs.shape[0])
        uncertainty = entropy / max_entropy
    else:
        uncertainty = 1 - probs.max(axis=0)
    
    # Power scaling to enhance contrast
    uncertainty_scaled = np.power(uncertainty, 0.5)
    uncertainty_uint8 = (uncertainty_scaled * 255).astype(np.uint8)
    
    thermal = cv2.applyColorMap(uncertainty_uint8, cv2.COLORMAP_MAGMA)
    thermal = cv2.cvtColor(thermal, cv2.COLOR_BGR2RGB)
    
    return thermal, uncertainty_scaled

if probs is not None:
    thermal_fog, uncertainty = create_thermal_fog(probs, method='entropy')
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(img)
    axes[0].set_title('Original Input', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(thermal_fog)
    axes[1].set_title('Style 2: THERMAL FOG\n"Where danger lurks"', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    # Overlay version
    axes[2].imshow(img)
    axes[2].imshow(thermal_fog, alpha=0.65)
    axes[2].set_title(f'Thermal Overlay\nMean Uncertainty: {uncertainty.mean():.3f}', fontsize=12, fontweight='bold')
    axes[2].axis('off')
    
    plt.suptitle('Style 2: Thermal Fog - Entropy-based Uncertainty', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if output_dir:
        save_path = output_dir / 'style2_thermal_fog.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    print(f"  Uncertainty stats: min={uncertainty.min():.3f}, max={uncertainty.max():.3f}, mean={uncertainty.mean():.3f}")
else:
    print("Run inference first (Section 3)")

### Style 3: GLOWING GHOST
> *"The model's gaze" - Attention visualization*

LayerCAM reveals which image regions most strongly influence the model's prediction for a specific class. The target glows from within.

In [ ]:
# 4.3 Style 3: Glowing Ghost (LayerCAM)

from pytorch_grad_cam import LayerCAM
from pytorch_grad_cam.utils.model_targets import SemanticSegmentationTarget

def create_glowing_ghost(grayscale_cam, img_normalized, colormap=cv2.COLORMAP_TURBO, alpha=0.7, enhance=True):
    """Create glowing overlay with histogram equalization."""
    if enhance:
        p2, p98 = np.percentile(grayscale_cam, (2, 98))
        cam_enhanced = np.clip((grayscale_cam - p2) / (p98 - p2 + 1e-8), 0, 1)
    else:
        cam_enhanced = grayscale_cam
    
    cam_uint8 = (cam_enhanced * 255).astype(np.uint8)
    heatmap = cv2.applyColorMap(cam_uint8, colormap)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    heatmap = heatmap.astype(np.float32) / 255.0
    
    blended = (1 - alpha) * img_normalized + alpha * heatmap
    blended = np.clip(blended, 0, 1)
    
    return (blended * 255).astype(np.uint8), cam_enhanced

# Class selector
class_selector = widgets.Dropdown(
    options=[(name, i) for i, name in enumerate(CLASS_NAMES)],
    value=0,
    description='Target Class:',
    style={'description_width': '100px'}
)

run_ghost_btn = widgets.Button(description='Generate Glowing Ghost', button_style='info', icon='eye')
ghost_output = widgets.Output()

def run_glowing_ghost(b):
    with ghost_output:
        clear_output(wait=True)
        
        if pred_mask is None:
            print("Run inference first (Section 3)")
            return
        
        target_class = class_selector.value
        class_mask = (pred_mask == target_class)
        coverage = class_mask.sum() / class_mask.size * 100
        
        if coverage < 0.1:
            print(f"Warning: {CLASS_NAMES[target_class]} has very low coverage ({coverage:.2f}%)")
        
        print(f"Generating Glowing Ghost for {CLASS_NAMES[target_class]} ({coverage:.1f}% coverage)...")
        
        # Target layers
        multi_layers = [
            wrapped_net.backbone.stage4[-1].branches[0][-1],
            wrapped_net.backbone.stage4[-1].branches[3][-1],
        ]
        
        targets = [SemanticSegmentationTarget(target_class, class_mask)]
        
        cam = LayerCAM(model=wrapped_net, target_layers=multi_layers)
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
        
        ghost_turbo, cam_enhanced = create_glowing_ghost(grayscale_cam, img_normalized, cv2.COLORMAP_TURBO)
        ghost_inferno, _ = create_glowing_ghost(grayscale_cam, img_normalized, cv2.COLORMAP_INFERNO)
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        axes[0].imshow(img)
        axes[0].set_title('Original Input', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        
        axes[1].imshow(ghost_turbo)
        axes[1].set_title(f'Style 3: GLOWING GHOST (Turbo)\nClass: {CLASS_NAMES[target_class]}', fontsize=12, fontweight='bold')
        axes[1].axis('off')
        
        axes[2].imshow(ghost_inferno)
        axes[2].set_title('Glowing Ghost (Inferno)\nDarker, dramatic', fontsize=12, fontweight='bold')
        axes[2].axis('off')
        
        plt.suptitle(f'Style 3: Glowing Ghost - LayerCAM for {CLASS_NAMES[target_class]}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        if output_dir:
            save_path = output_dir / f'style3_glowing_ghost_{CLASS_NAMES[target_class].lower()}.png'
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Saved: {save_path}")
        
        plt.show()
        
        del cam
        torch.cuda.empty_cache()

run_ghost_btn.on_click(run_glowing_ghost)

print("Select a class and click 'Generate Glowing Ghost':")
display(widgets.HBox([class_selector, run_ghost_btn]))
display(ghost_output)

### Style 4: STATIC NOISE
> *"Raw pixel evidence" - Attribution visualization*

Integrated Gradients reveals which individual pixels contributed positively (orange) or negatively (purple) to the model's prediction. Raw, unsmoothed, forensic-level detail.

In [ ]:
# 4.4 Style 4: Static Noise (Integrated Gradients)

from captum.attr import IntegratedGradients

class IGWrapper(nn.Module):
    def __init__(self, model, target_class):
        super().__init__()
        self.model = model
        self.target_class = target_class

    def forward(self, x):
        output = self.model({'images': x})
        logits = output.get('pred', list(output.values())[0])
        return logits[:, self.target_class, :, :].sum(dim=(-1, -2))

# Class selector for IG
ig_class_selector = widgets.Dropdown(
    options=[(name, i) for i, name in enumerate(CLASS_NAMES)],
    value=0,
    description='Target Class:',
    style={'description_width': '100px'}
)

run_ig_btn = widgets.Button(description='Generate Static Noise', button_style='warning', icon='bolt')
ig_output = widgets.Output()

def run_static_noise(b):
    with ig_output:
        clear_output(wait=True)
        
        if input_tensor is None:
            print("Run inference first (Section 3)")
            return
        
        target_class = ig_class_selector.value
        print(f"Generating Static Noise for {CLASS_NAMES[target_class]}...")
        print("(This may take a minute)")
        
        # Resize for memory
        resize_dim = 256
        resize_tf = transforms.Resize((resize_dim, resize_dim), antialias=True)
        resized_input = resize_tf(input_tensor)
        resized_img = img.resize((resize_dim, resize_dim), Image.LANCZOS)
        resized_img_norm = np.array(resized_img).astype(np.float32) / 255.0
        baseline = torch.zeros_like(resized_input)
        
        torch.cuda.empty_cache()
        ig_model = IGWrapper(net, target_class).eval()
        ig = IntegratedGradients(ig_model)
        
        attributions = ig.attribute(
            resized_input,
            baselines=baseline,
            target=None,
            n_steps=50,
            internal_batch_size=1
        )
        
        attr_np = attributions.squeeze().cpu().detach().numpy()
        if len(attr_np.shape) == 3:
            attr_signed = attr_np.sum(axis=0)
        else:
            attr_signed = attr_np
        
        # Percentile normalization
        p1, p99 = np.percentile(np.abs(attr_signed), (1, 99))
        attr_norm = np.clip(attr_signed / (p99 + 1e-8), -1, 1)
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        axes[0].imshow(resized_img)
        axes[0].set_title('Original (256x256)', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        
        # RdBu overlay
        axes[1].imshow(resized_img_norm)
        im = axes[1].imshow(attr_norm, cmap='RdBu_r', vmin=-1, vmax=1, alpha=0.6)
        axes[1].set_title(f'Style 4: STATIC NOISE\nClass: {CLASS_NAMES[target_class]}', fontsize=12, fontweight='bold')
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1], fraction=0.046)
        
        # Absolute magnitude
        axes[2].imshow(resized_img_norm)
        axes[2].imshow(np.abs(attr_norm), cmap='hot', vmin=0, vmax=1, alpha=0.65)
        axes[2].set_title('Absolute Attribution\n(Magnitude only)', fontsize=12, fontweight='bold')
        axes[2].axis('off')
        
        plt.suptitle(f'Style 4: Static Noise - Integrated Gradients for {CLASS_NAMES[target_class]}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        if output_dir:
            save_path = output_dir / f'style4_static_noise_{CLASS_NAMES[target_class].lower()}.png'
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Saved: {save_path}")
        
        plt.show()
        torch.cuda.empty_cache()

run_ig_btn.on_click(run_static_noise)

print("Select a class and click 'Generate Static Noise':")
display(widgets.HBox([ig_class_selector, run_ig_btn]))
display(ig_output)

### Summary: All 4 Visual Styles

In [ ]:
# 4.5 Summary Grid - All 4 Styles

def generate_summary_grid():
    if pred_mask is None:
        print("Run inference first (Section 3)")
        return
    
    print("Generating summary grid with all 4 visual styles...")
    
    # Generate all styles
    cartoon = create_cartoon_prediction(pred_mask, CARTOON_COLORS, add_edges=True)
    thermal, _ = create_thermal_fog(probs, method='entropy')
    
    # Find best class for attention
    class_coverages = [(pred_mask == i).sum() / pred_mask.size for i in range(4)]
    non_bg = [(i, c) for i, c in enumerate(class_coverages) if i != 3 and c > 0.01]
    target_class = max(non_bg, key=lambda x: x[1])[0] if non_bg else 0
    
    # LayerCAM
    from pytorch_grad_cam import LayerCAM
    multi_layers = [
        wrapped_net.backbone.stage4[-1].branches[0][-1],
        wrapped_net.backbone.stage4[-1].branches[3][-1],
    ]
    class_mask = (pred_mask == target_class)
    targets = [SemanticSegmentationTarget(target_class, class_mask)]
    cam = LayerCAM(model=wrapped_net, target_layers=multi_layers)
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
    ghost, _ = create_glowing_ghost(grayscale_cam, img_normalized, cv2.COLORMAP_TURBO)
    del cam
    
    # Create grid
    fig, axes = plt.subplots(2, 3, figsize=(20, 14))
    
    axes[0, 0].imshow(img)
    axes[0, 0].set_title('Original Input', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(cartoon)
    axes[0, 1].set_title('1. CARTOON BLOCKS\n"This is my claim"', fontsize=12, fontweight='bold', color='#2D2D2D')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(img)
    axes[0, 2].imshow(thermal, alpha=0.65)
    axes[0, 2].set_title('2. THERMAL FOG\n"Where danger lurks"', fontsize=12, fontweight='bold', color='#D35400')
    axes[0, 2].axis('off')
    
    axes[1, 0].imshow(ghost)
    axes[1, 0].set_title(f'3. GLOWING GHOST\n"The model\'s gaze" ({CLASS_NAMES[target_class]})', fontsize=12, fontweight='bold', color='#2980B9')
    axes[1, 0].axis('off')
    
    # Placeholder for Static Noise (would need IG computation)
    axes[1, 1].imshow(img)
    axes[1, 1].text(0.5, 0.5, '4. STATIC NOISE\n(Run individually\nfor full detail)', 
                   transform=axes[1, 1].transAxes, ha='center', va='center',
                   fontsize=12, fontweight='bold', color='white',
                   bbox=dict(boxstyle='round', facecolor='#8E44AD', alpha=0.8))
    axes[1, 1].set_title('4. STATIC NOISE\n"Raw pixel evidence"', fontsize=12, fontweight='bold', color='#8E44AD')
    axes[1, 1].axis('off')
    
    # Legend
    axes[1, 2].axis('off')
    legend_text = f"""4 VISUAL LANGUAGES

1. CARTOON BLOCKS
   Solid colors + black edges
   Shows: Model's prediction

2. THERMAL FOG
   Magma colormap (entropy)
   Shows: Uncertainty zones

3. GLOWING GHOST
   LayerCAM attention
   Shows: Where model looks

4. STATIC NOISE
   Integrated Gradients
   Shows: Pixel evidence

Image: {current_name}
Target: {CLASS_NAMES[target_class]}"""
    axes[1, 2].text(0.05, 0.95, legend_text, fontsize=10, va='top',
                   fontfamily='monospace', transform=axes[1, 2].transAxes,
                   bbox=dict(boxstyle='round', facecolor='#F8F9FA', edgecolor='#DEE2E6', alpha=0.95))
    axes[1, 2].set_title('Visual Style Guide', fontsize=14, fontweight='bold')
    
    plt.suptitle('The Segmentation Detective: 4 Visual Languages', fontsize=18, fontweight='bold')
    plt.tight_layout()
    
    if output_dir:
        save_path = output_dir / 'summary_4_styles.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    torch.cuda.empty_cache()

generate_summary_grid()

---
## Section 5: Deep Dive XAI

For those who want to investigate further...

### 5.1 GradCAM Across Network Layers

See how attention evolves from early layers (fine edges) to late layers (semantic understanding).

In [ ]:
# 5.1 GradCAM across layers

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

gradcam_class_selector = widgets.Dropdown(
    options=[(name, i) for i, name in enumerate(CLASS_NAMES)],
    value=0,
    description='Target Class:',
    style={'description_width': '100px'}
)

run_gradcam_btn = widgets.Button(description='Run GradCAM Comparison', button_style='success', icon='layer-group')
gradcam_output = widgets.Output()

def run_gradcam_layers(b):
    with gradcam_output:
        clear_output(wait=True)
        
        if pred_mask is None:
            print("Run inference first (Section 3)")
            return
        
        target_class = gradcam_class_selector.value
        class_mask = (pred_mask == target_class)
        coverage = class_mask.sum() / class_mask.size * 100
        
        print(f"GradCAM for {CLASS_NAMES[target_class]} ({coverage:.1f}% coverage)...")
        
        target_layers = {
            'Stage2-High': [wrapped_net.backbone.stage2[-1].branches[0][-1]],
            'Stage2-Low': [wrapped_net.backbone.stage2[-1].branches[1][-1]],
            'Stage3-High': [wrapped_net.backbone.stage3[-1].branches[0][-1]],
            'Stage4-High': [wrapped_net.backbone.stage4[-1].branches[0][-1]],
            'Stage4-Low': [wrapped_net.backbone.stage4[-1].branches[3][-1]],
        }
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        axes[0].imshow(img)
        axes[0].set_title('Original', fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        targets = [SemanticSegmentationTarget(target_class, class_mask)]
        
        for idx, (layer_name, layers) in enumerate(target_layers.items(), start=1):
            try:
                cam = GradCAM(model=wrapped_net, target_layers=layers)
                grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
                cam_image = show_cam_on_image(img_normalized, grayscale_cam, use_rgb=True)
                
                axes[idx].imshow(cam_image)
                axes[idx].set_title(layer_name, fontsize=11, fontweight='bold')
                axes[idx].axis('off')
                del cam
            except Exception as e:
                axes[idx].text(0.5, 0.5, f'Error', ha='center', va='center')
                axes[idx].axis('off')
        
        plt.suptitle(f'GradCAM: {CLASS_NAMES[target_class]} Across HRNet Layers', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        if output_dir:
            save_path = output_dir / f'gradcam_layers_{CLASS_NAMES[target_class].lower()}.png'
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Saved: {save_path}")
        
        plt.show()
        torch.cuda.empty_cache()

run_gradcam_btn.on_click(run_gradcam_layers)

print("Compare GradCAM across network layers:")
display(widgets.HBox([gradcam_class_selector, run_gradcam_btn]))
display(gradcam_output)

### 5.2 Confidence Maps

In [ ]:
# 5.2 Confidence Maps

def visualize_confidence_maps():
    if probs is None:
        print("Run inference first (Section 3)")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    axes = axes.flatten()
    
    for class_id, (ax, class_name) in enumerate(zip(axes, CLASS_NAMES)):
        prob_map = probs[class_id]
        im = ax.imshow(prob_map, cmap='viridis', vmin=0, vmax=1)
        ax.set_title(f'{class_name} Probability', fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)
    
    plt.suptitle('Per-Class Probability Maps (Softmax)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if output_dir:
        save_path = output_dir / 'confidence_maps.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()

visualize_confidence_maps()

### 5.3 ScoreCAM (Gradient-Free)

In [ ]:
# 5.3 ScoreCAM

from pytorch_grad_cam import ScoreCAM

scorecam_class_selector = widgets.Dropdown(
    options=[(name, i) for i, name in enumerate(CLASS_NAMES)],
    value=0,
    description='Target Class:',
    style={'description_width': '100px'}
)

run_scorecam_btn = widgets.Button(description='Run ScoreCAM (Slow)', button_style='danger', icon='stopwatch')
scorecam_output = widgets.Output()

def run_scorecam(b):
    with scorecam_output:
        clear_output(wait=True)
        
        if pred_mask is None:
            print("Run inference first (Section 3)")
            return
        
        target_class = scorecam_class_selector.value
        print(f"Running ScoreCAM for {CLASS_NAMES[target_class]}...")
        print("(This is gradient-free and slower)")
        
        # Resize for memory
        resize_dim = 256
        resize_transform = transforms.Resize((resize_dim, resize_dim), antialias=True)
        resized_input = resize_transform(input_tensor)
        resized_img = img.resize((resize_dim, resize_dim), Image.LANCZOS)
        resized_img_norm = np.array(resized_img).astype(np.float32) / 255.0
        
        class_mask = (pred_mask == target_class)
        mask_pil = transforms.ToPILImage()(class_mask.astype(np.uint8) * 255)
        resized_mask = np.array(resize_transform(mask_pil)) > 0
        
        score_cam_layer = [wrapped_net.backbone.stage4[-1].branches[0][-1]]
        targets = [SemanticSegmentationTarget(target_class, resized_mask)]
        
        torch.cuda.empty_cache()
        gc.collect()
        
        try:
            cam = ScoreCAM(model=wrapped_net, target_layers=score_cam_layer)
            grayscale_cam = cam(input_tensor=resized_input, targets=targets)[0]
            cam_image = show_cam_on_image(resized_img_norm, grayscale_cam, use_rgb=True)
            
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            axes[0].imshow(resized_img)
            axes[0].set_title('Input (256x256)', fontweight='bold')
            axes[0].axis('off')
            
            axes[1].imshow(resized_mask, cmap='gray')
            axes[1].set_title(f'{CLASS_NAMES[target_class]} Mask', fontweight='bold')
            axes[1].axis('off')
            
            axes[2].imshow(cam_image)
            axes[2].set_title(f'ScoreCAM: {CLASS_NAMES[target_class]}', fontweight='bold')
            axes[2].axis('off')
            
            plt.suptitle('ScoreCAM: Gradient-Free Validation', fontsize=16, fontweight='bold')
            plt.tight_layout()
            
            if output_dir:
                save_path = output_dir / f'scorecam_{CLASS_NAMES[target_class].lower()}.png'
                plt.savefig(save_path, dpi=150, bbox_inches='tight')
                print(f"Saved: {save_path}")
            
            plt.show()
            del cam
            
        except Exception as e:
            print(f"Error: {e}")
        
        torch.cuda.empty_cache()

run_scorecam_btn.on_click(run_scorecam)

print("ScoreCAM (gradient-free, slower but more faithful):")
display(widgets.HBox([scorecam_class_selector, run_scorecam_btn]))
display(scorecam_output)

---
## Section 6: Misc - NYC Ground Truth Comparison

Compare model predictions against official NYC sidewalk data.

In [ ]:
# 6.1 NYC Ground Truth Comparison

import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union
from pyproj import Transformer

def compare_with_nyc_gt(bbox):
    """
    Compare model predictions with NYC Open Data sidewalk ground truth.
    bbox: [south, north, west, east]
    """
    if pred_mask is None:
        print("Run inference first!")
        return
    
    south, north, west, east = bbox
    buffer_deg = 0.0005
    south_exp, north_exp = south - buffer_deg, north + buffer_deg
    west_exp, east_exp = west - buffer_deg, east + buffer_deg
    
    bbox_poly = box(west_exp, south_exp, east_exp, north_exp)
    
    print("Fetching NYC sidewalk ground truth...")
    
    base_url = "https://data.cityofnewyork.us/resource/52n9-sdep.geojson"
    spatial_query = f"$where=within_box(the_geom, {north_exp}, {west_exp}, {south_exp}, {east_exp})"
    url = f"{base_url}?{spatial_query}&$limit=10000"
    
    try:
        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            data = response.json()
            features = data.get('features', [])
            print(f"  Found {len(features)} sidewalk features")
            
            if len(features) > 0:
                gt_gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
                
                # Rasterize
                from rasterio.features import rasterize
                from affine import Affine
                
                h, w = pred_mask.shape
                transform = Affine.translation(west, north) * Affine.scale((east - west) / w, (south - north) / h)
                
                gt_raster = rasterize(
                    [(geom, 1) for geom in gt_gdf.geometry],
                    out_shape=(h, w),
                    transform=transform,
                    fill=0,
                    dtype=np.uint8
                )
                
                # Compare
                pred_sidewalk = (pred_mask == 0).astype(np.uint8)
                
                intersection = np.logical_and(pred_sidewalk, gt_raster).sum()
                union = np.logical_or(pred_sidewalk, gt_raster).sum()
                pred_sum = pred_sidewalk.sum()
                gt_sum = gt_raster.sum()
                
                iou = intersection / union if union > 0 else 0
                precision = intersection / pred_sum if pred_sum > 0 else 0
                recall = intersection / gt_sum if gt_sum > 0 else 0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                
                # Visualize
                fig, axes = plt.subplots(2, 2, figsize=(14, 14))
                
                axes[0, 0].imshow(img)
                axes[0, 0].set_title('Input Image', fontsize=14, fontweight='bold')
                axes[0, 0].axis('off')
                
                axes[0, 1].imshow(pred_sidewalk, cmap='Blues')
                axes[0, 1].set_title(f'Model: Sidewalk\n{pred_sum} pixels', fontsize=12, fontweight='bold')
                axes[0, 1].axis('off')
                
                axes[1, 0].imshow(gt_raster, cmap='Greens')
                axes[1, 0].set_title(f'NYC Ground Truth\n{gt_sum} pixels', fontsize=12, fontweight='bold')
                axes[1, 0].axis('off')
                
                # Overlay
                overlay = np.zeros((*pred_mask.shape, 3))
                overlay[pred_sidewalk == 1, 0] = 1  # Red = model only
                overlay[gt_raster == 1, 1] = 1      # Green = GT only
                overlay[np.logical_and(pred_sidewalk, gt_raster), :] = [1, 1, 0]  # Yellow = match
                
                axes[1, 1].imshow(img)
                axes[1, 1].imshow(overlay, alpha=0.6)
                axes[1, 1].set_title(f'Comparison\nIoU: {iou:.1%}, Recall: {recall:.1%}', fontsize=12, fontweight='bold')
                axes[1, 1].axis('off')
                
                plt.suptitle('NYC Ground Truth Comparison', fontsize=16, fontweight='bold')
                plt.tight_layout()
                
                if output_dir:
                    save_path = output_dir / 'nyc_gt_comparison.png'
                    plt.savefig(save_path, dpi=150, bbox_inches='tight')
                    print(f"Saved: {save_path}")
                
                plt.show()
                
                print(f"\nMetrics:")
                print(f"  IoU:       {iou:.1%}")
                print(f"  Precision: {precision:.1%}")
                print(f"  Recall:    {recall:.1%}")
                print(f"  F1:        {f1:.1%}")
            else:
                print("No ground truth features found in this area.")
    except Exception as e:
        print(f"Error fetching ground truth: {e}")

# Only run if using NYC tile
print("NYC Ground Truth comparison (only works for NYC tiles):")
print("If using Washington Square Park tile, run the cell below.")

In [ ]:
# Run NYC GT comparison for Washington Square Park
# Default bbox for WSP area
wsp_bbox = [40.7290, 40.7326, -73.9998, -73.9948]  # [south, north, west, east]

# Uncomment to run:
# compare_with_nyc_gt(wsp_bbox)

---
## Section 7: Export All Results

In [ ]:
# 7.1 List all saved files

def list_saved_files():
    if output_dir and output_dir.exists():
        files = list(output_dir.glob('*.png'))
        print(f"Saved files in {output_dir}:")
        print("=" * 50)
        for f in sorted(files):
            size_kb = f.stat().st_size / 1024
            print(f"  {f.name} ({size_kb:.1f} KB)")
        print(f"\nTotal: {len(files)} files")
    else:
        print("No output directory set. Run Section 2 first.")

list_saved_files()

In [ ]:
# 7.2 Create ZIP for download

def create_download_zip():
    if output_dir and output_dir.exists():
        zip_path = output_dir.parent / f"{output_dir.name}.zip"
        
        import shutil
        shutil.make_archive(str(output_dir), 'zip', output_dir)
        
        print(f"Created: {zip_path}")
        print(f"Size: {zip_path.stat().st_size / 1024:.1f} KB")
        print(f"\nDownload from: {zip_path}")
        
        # For Kaggle, create download link
        from IPython.display import FileLink
        return FileLink(str(zip_path))
    else:
        print("No files to zip.")

create_download_zip()

---
## Summary & Key Findings

### What We Learned

1. **Cartoon Blocks** reveal the model's confident claims - sharp boundaries show clear segmentation

2. **Thermal Fog** exposes uncertainty hotspots - typically at class boundaries where the model hedges its bets

3. **Glowing Ghost** shows attention patterns - the model focuses on texture and edges, not just location

4. **Static Noise** provides forensic evidence - individual pixels that push the model toward or away from a class

### The Bigger Picture

XAI isn't just about debugging - it's about **trust**. When urban planners use Tile2Net to map pedestrian infrastructure, they need to know:
- Where is the model confident?
- Where might it be wrong?
- What features is it actually using?

These 4 visual styles answer those questions in ways that are **intuitive** and **actionable**.

---

*The Segmentation Detective - NYU CS-GY 9223 Fall 2024*